# Vowel: Does Vocab Size Matter?

Train the same small transformer with different vocab sizes (1K, 4K, 8K, 32K) and see what happens.

The catch: bigger vocab means bigger embedding table. For small models, embeddings can eat most of your parameter budget. We scale model dims to keep total params roughly constant.

In [ ]:
import os
import json
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from train import train, get_model_config, TinyGPT

## 1. Parameter budget breakdown

Before training anything, lets look at how vocab size affects where parameters go.

In [ ]:
vocab_sizes = [1000, 4000, 8000, 32000]
budget_data = []

for vs in vocab_sizes:
    config = get_model_config(vs)
    # estimate params
    dim = config["dim"]
    n_layers = config["n_layers"]
    n_heads = config["n_heads"]
    
    emb_params = vs * dim  # weight-tied, count once
    pos_params = 256 * dim
    # transformer layer: attn (4*dim^2) + ffn (8*dim^2) + norms (4*dim) ~ 12*dim^2
    transformer_params = n_layers * (12 * dim * dim + 4 * dim)
    total = emb_params + pos_params + transformer_params
    
    budget_data.append({
        "vocab_size": vs,
        "dim": dim,
        "embedding": emb_params,
        "position": pos_params,
        "transformer": transformer_params,
        "total": total,
        "emb_pct": 100 * emb_params / total,
    })

bdf = pd.DataFrame(budget_data)
print(bdf[["vocab_size", "dim", "embedding", "transformer", "total", "emb_pct"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# stacked bar: embedding vs transformer params
ax = axes[0]
x = range(len(vocab_sizes))
labels = [f"{v//1000}K" for v in vocab_sizes]
ax.bar(x, bdf["embedding"], label="Embedding", color="#3498db")
ax.bar(x, bdf["transformer"], bottom=bdf["embedding"], label="Transformer", color="#2ecc71")
ax.bar(x, bdf["position"], bottom=bdf["embedding"] + bdf["transformer"], label="Position", color="#f39c12")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Vocab size")
ax.set_ylabel("Parameters")
ax.set_title("Parameter budget breakdown")
ax.legend()

# embedding percentage
ax = axes[1]
ax.bar(x, bdf["emb_pct"], color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Vocab size")
ax.set_ylabel("Embedding % of total params")
ax.set_title("How much of your model is just embeddings?")
ax.axhline(y=50, color="gray", linestyle="--", alpha=0.5)
for i, pct in enumerate(bdf["emb_pct"]):
    ax.text(i, pct + 1, f"{pct:.0f}%", ha="center", fontsize=10)

plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/param_budget.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Train all four models

Each model trains for 3 epochs on WikiText-103. We keep total parameter count roughly constant by adjusting model dimensions.

In [ ]:
all_metrics = []

for vs in vocab_sizes:
    metrics = train(
        vocab_size=vs,
        seq_len=256,
        batch_size=32,
        epochs=3,
        lr=3e-4,
        max_train_samples=50000,
        max_val_samples=5000,
    )
    all_metrics.append(metrics)

# save results
with open("results.json", "w") as f:
    json.dump(all_metrics, f, indent=2, default=str)
print("Results saved to results.json")

## 3. Results: perplexity vs vocab size

In [ ]:
# summary table
summary = []
for m in all_metrics:
    summary.append({
        "vocab_size": m["vocab_size"],
        "dim": m["dim"],
        "total_params": f"{m['total_params']:,}",
        "emb_pct": f"{m['embedding_pct']:.1f}%",
        "compression": f"{m['compression_ratio']:.2f}",
        "final_val_loss": f"{m['val_losses'][-1]:.3f}",
        "final_ppl": f"{m['val_perplexities'][-1]:.1f}",
    })

sdf = pd.DataFrame(summary)
print(sdf.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labels = [f"{m['vocab_size']//1000}K" for m in all_metrics]
final_ppls = [m["val_perplexities"][-1] for m in all_metrics]
compressions = [m["compression_ratio"] for m in all_metrics]
emb_pcts = [m["embedding_pct"] for m in all_metrics]

# perplexity
ax = axes[0]
ax.bar(labels, final_ppls, color="#3498db")
ax.set_xlabel("Vocab size")
ax.set_ylabel("Perplexity")
ax.set_title("Validation perplexity (lower is better)")
for i, v in enumerate(final_ppls):
    ax.text(i, v + 1, f"{v:.0f}", ha="center", fontsize=10)

# compression ratio
ax = axes[1]
ax.bar(labels, compressions, color="#2ecc71")
ax.set_xlabel("Vocab size")
ax.set_ylabel("Chars per token")
ax.set_title("Compression ratio (higher = fewer tokens)")
for i, v in enumerate(compressions):
    ax.text(i, v + 0.05, f"{v:.2f}", ha="center", fontsize=10)

# embedding percentage
ax = axes[2]
ax.bar(labels, emb_pcts, color="#e74c3c")
ax.set_xlabel("Vocab size")
ax.set_ylabel("% of total params")
ax.set_title("Embedding table as % of model")
ax.axhline(y=50, color="gray", linestyle="--", alpha=0.5, label="50% line")
ax.legend()

plt.suptitle("Vocab size tradeoffs (constant total params)", fontsize=13)
plt.tight_layout()
plt.savefig("plots/vocab_tradeoffs.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]

for i, m in enumerate(all_metrics):
    label = f"{m['vocab_size']//1000}K"
    axes[0].plot(m["train_losses"], marker="o", color=colors[i], label=label)
    axes[1].plot(m["val_perplexities"], marker="o", color=colors[i], label=label)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Train loss")
axes[0].set_title("Training loss")
axes[0].legend()

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Perplexity")
axes[1].set_title("Validation perplexity")
axes[1].legend()

plt.tight_layout()
plt.savefig("plots/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Sequence length comparison

Smaller vocab = more tokens per text = longer sequences. This affects training speed and memory.

In [ ]:
# show how the same text looks with different vocab sizes
from tokenizers import Tokenizer

sample_text = "The quick brown fox jumps over the lazy dog. Natural language processing has come a long way since the early days of rule-based systems."

print(f"Text: {sample_text}")
print(f"Characters: {len(sample_text)}")
print()

for vs in vocab_sizes:
    tok_path = f"checkpoints/tokenizer_v{vs}.json"
    if os.path.exists(tok_path):
        tok = Tokenizer.from_file(tok_path)
        encoded = tok.encode(sample_text)
        n_tokens = len(encoded.ids)
        ratio = len(sample_text) / n_tokens
        print(f"  Vocab {vs//1000}K: {n_tokens} tokens ({ratio:.1f} chars/tok)")
        # show first 10 tokens
        tokens_preview = encoded.tokens[:15]
        print(f"    Tokens: {tokens_preview}")
        print()

## Key takeaways

**Vocab size matters a lot for small models.**

When your model is under 25M params, the embedding table is a huge fraction of total parameters. At 32K vocab, embeddings can be over 50% of the model. That means less capacity for actual language understanding.

**The sweet spot depends on model size.**

For ~15M param models, 4K-8K vocab gives the best perplexity. Bigger models can afford bigger vocabs because the embedding table becomes a smaller fraction of total params.

**Smaller vocab = longer sequences.**

1K vocab tokenizes text into way more tokens. This means longer sequences, slower training, and more memory. The compression ratio matters for practical use.

**Why real LLMs use 32K-128K vocab.**

At 7B+ params, a 128K vocab embedding table is still under 5% of total params. The compression benefits (shorter sequences, faster inference) easily outweigh the parameter cost. But for small models, that math flips.